# Drishti — OCR Spike (standalone, CPU-only)

Split out of `00_feasibility_spike_colab.ipynb` on purpose. Running PaddleOCR in a process
that already has the VLMs loaded **hard-kills the kernel**
(`AsyncIOLoopKernelRestarter: restarting kernel`) — the process dies, so you get no Python
traceback. Observed on Colab; the same collision applies anywhere. Two causes stack:

- **PyTorch and PaddlePaddle each bundle their own OpenMP runtime.** Co-loading both in one
  process is a well-known segfault source.
- **Memory.** The two VLMs are ~8.5 GB of weights, and Colab's free tier has ~12.7 GB of
  system RAM — little headroom once PaddlePaddle and its models load too.

So this notebook is deliberately minimal:
- **no torch, no transformers, no VLMs** — no OpenMP clash, no memory pressure
- CPU-only PaddleOCR (PP-OCR models are small)
- runs in a **fresh Colab runtime** or **locally in VS Code** — it needs no GPU

Verifies two load-bearing assumptions for Drishti:
1. Medicine mode — can OCR read a real strip's **drug name, EXP, MRP**? (`lang='en'`)
2. Read mode — can it read **Devanagari** (Marathi/Hindi)? (`lang='devanagari'`)

> **Run this in a fresh runtime** — not the one where §2a/§2b already loaded the VLMs.
> That collision is the whole reason this notebook is separate.
>
> Keep the VLM cells (`00_..._colab.ipynb` §2a/§2b) and the VizWiz baseline
> (`01_vizwiz_baseline.ipynb`) on Colab with a T4 GPU — those genuinely need the GPU.

In [ ]:
# CPU build: matches the eventual phone target, and CPU timings are the honest number
# for an on-device app. No GPU/CUDA needed anywhere in this notebook.
%pip install -q paddlepaddle paddleocr

import time
from importlib.metadata import version
from pathlib import Path

import numpy as np
from PIL import Image

print('paddleocr version:', version('paddleocr'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/9

## 1. Load your medicine-strip photo

**Colab:** leave `IMAGE_PATHS` empty — an upload widget appears.
**Local (VS Code):** set `IMAGE_PATHS` to your photo path(s).

In [ ]:
# Point these at real photos: back of a medicine strip (name + EXP + MRP visible).
# Leave empty on Colab to get an upload widget instead.
IMAGE_PATHS = [
    # r'C:\Users\devgu\Downloads\strip1.jpg',
]

images, labels = [], []

if IMAGE_PATHS:
    for p in IMAGE_PATHS:
        path = Path(p)
        if not path.exists():
            raise FileNotFoundError(f'not found: {path}')
        images.append(Image.open(path).convert('RGB'))
        labels.append(path.name)
else:
    try:
        from google.colab import files
        for name in files.upload():
            images.append(Image.open(name).convert('RGB'))
            labels.append(name)
    except ImportError:
        raise SystemExit('Not on Colab — set IMAGE_PATHS above to your photo(s).')

print(f'{len(images)} image(s) loaded:', ', '.join(labels))
for img, name in zip(images, labels):
    print(f'  {name}: {img.size[0]}x{img.size[1]}')

Saving IMG_20260801_203248259.jpg to IMG_20260801_203248259.jpg
Saving IMG_20260801_203256207.jpg to IMG_20260801_203256207.jpg
2 image(s) loaded: IMG_20260801_203248259.jpg, IMG_20260801_203256207.jpg
  IMG_20260801_203248259.jpg: 4080x3072
  IMG_20260801_203256207.jpg: 4080x3072


## 2. Run OCR — English and Devanagari

In [ ]:
from paddleocr import PaddleOCR

# 12 MP phone photos are far more than OCR needs, cost RAM (PaddleOCR 3.x has a reported
# CPU memory blowup on large inputs), and a real phone app would downscale before inference
# anyway. 1600px on the long side keeps strip print legible.
MAX_SIDE = 1600


def downscale(img, max_side=MAX_SIDE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    s = max_side / max(w, h)
    return img.resize((int(w * s), int(h * s)), Image.LANCZOS)


def extract_lines(result):
    """Normalize PaddleOCR output to [(confidence, text)].

    Handles both result shapes so this survives a version bump:
      3.x .predict() -> objects/dicts carrying `rec_texts` + `rec_scores`
      2.x .ocr()     -> nested [[bbox, (text, score)], ...]
    """
    lines = []
    for page in result or []:
        texts = scores = None

        if isinstance(page, dict):
            texts, scores = page.get('rec_texts'), page.get('rec_scores')
        else:
            texts = getattr(page, 'rec_texts', None)
            scores = getattr(page, 'rec_scores', None)
            if texts is None and hasattr(page, 'json'):
                blob = page.json
                blob = blob.get('res', blob) if isinstance(blob, dict) else {}
                texts, scores = blob.get('rec_texts'), blob.get('rec_scores')

        if texts is not None:
            scores = scores if scores is not None else [float('nan')] * len(texts)
            lines.extend(zip(scores, texts))
            continue

        if isinstance(page, list):          # 2.x layout
            for item in page:
                try:
                    _bbox, (text, score) = item
                    lines.append((score, text))
                except (TypeError, ValueError):
                    continue
    return lines


def _kwarg_variants(lang):
    """Config combinations to try, best-first.

    Two known 3.7.0 problems drive this:
      * enable_mkldnn=False dodges the PIR/oneDNN CPU crash
        ("ConvertPirAttribute2RuntimeAttribute not support") -- Paddle issue #77340.
      * PP-OCRv6 (the current default) covers English/Chinese/Japanese + 46 Latin-script
        languages only. Devanagari lives in the PP-OCRv5/v3 language-specific groupings,
        so an explicit older ocr_version is required for Marathi/Hindi.
    """
    for ver in (None, 'PP-OCRv5', 'PP-OCRv4', 'PP-OCRv3'):
        for mkldnn in (False, None):
            kw = {'lang': lang}
            if ver is not None:
                kw['ocr_version'] = ver
            if mkldnn is not None:
                kw['enable_mkldnn'] = mkldnn
            yield kw


def run_paddle(imgs, lang):
    """Try config variants until one constructs AND predicts. Returns (lines, secs, config)."""
    arrays = [np.array(downscale(img)) for img in imgs]
    last_err = None

    for kw in _kwarg_variants(lang):
        tag = f"ocr_version={kw.get('ocr_version', 'default')}, mkldnn={kw.get('enable_mkldnn', 'default')}"
        try:
            ocr = PaddleOCR(**kw)
            out, t0 = [], time.time()
            for arr in arrays:
                out.extend(extract_lines(ocr.predict(arr) if hasattr(ocr, 'predict') else ocr.ocr(arr)))
            return out, time.time() - t0, tag
        except Exception as e:
            last_err = e
            print(f'    tried {tag} -> {type(e).__name__}')

    raise last_err


results = {}
for lang in ('en', 'devanagari'):
    print(f'\n### lang="{lang}"')
    try:
        lines, secs, cfg = run_paddle(images, lang)
        results[lang] = lines
        print(f'=== OK via [{cfg}] — {secs:.1f}s (CPU), {len(lines)} lines ===')
        for conf, text in lines:
            print(f'  {conf:.2f}  {text}')
    except Exception as e:
        results[lang] = []
        print(f'=== all variants FAILED: {type(e).__name__}: {e} ===')

# medicine mode reads Latin-script fields (drug name / EXP / MRP)
ocr_lines = results.get('en', [])
ocr_text = ' '.join(text for _, text in ocr_lines)
print(f'\nocr_text: {ocr_text[:200]}')


### lang="en"


/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_doc_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('UVDoc', None, None)
Using official model (UVDoc), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/UVDoc`.


...

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

=== OK via [ocr_version=default, mkldnn=False] — 104.3s (CPU), 55 lines ===
  0.96  INCL. OF ALL TAXES
  0.98  MFG.NOV.2024 EXP.OCT.2026
  0.91  ER 20 TABS.
  0.96  Paracetamol Tablets IP
  0.91  mol Tablets IP
  1.00  500mg
  0.57  p
  0.64  nd     e
  0.98  PARACIP-500-500
  1.00  CIP-500-500
  0.98  Antipyretic & Relief from Body Pain
  0.80  Taking mpue
  0.91  serious her com
  0.95  from Body Pain
  0.83  Taling more than daly dose of Paracetamol may cause
  1.00  Each uncoated tablet contains
  0.77  face noahad
  0.90  in brect
  0.59  sres (i
  0.91  Paracetamol IP
  0.98  500 mg
  0.92  el contains
  0.49  lmauy ,
  0.92  Stare below 
  0.93  500 mg
  0.98  Store below 30 C. Protect from light & moisture
  1.00  Excipients
  0.98  g.s
  0.78  Xesp out er react
  0.89  Keep out of reach of children. M. 1 106/14/2007
  0.93  Dasage Adults: 500 mg 1o 1000 mg
  0.81  g.s
  0.53  Of.SOCUL Horara
  0.94  mg 10 1000 mg
  0.92  Manutactured by: HShKinternalional, Piot No 5455
  0.98 

In [ ]:
# === Tesseract cross-check =================================================================
# Runs regardless of how PaddleOCR fared. Two reasons this earns its place:
#   1. Insurance -- Tesseract is apt-installable, pure CPU, and has no version drama, so this
#      notebook always produces *some* OCR output to reason about.
#   2. Comparing two independent engines on the same real photos is a legitimate evaluation
#      for the report, not just a fallback. Devanagari support here is a separate apt package
#      (hin/mar) rather than PaddleOCR's ocr_version maze.
# If PaddleOCR produced nothing for a script, the Tesseract text is promoted into `ocr_text`
# so the downstream cells still have something real to work on.

!apt-get -qq install -y tesseract-ocr tesseract-ocr-hin tesseract-ocr-mar > /dev/null 2>&1
%pip install -q pytesseract

import pytesseract

TESS_LANGS = {'eng': 'English', 'hin': 'Hindi', 'mar': 'Marathi'}
tess_text = {}

for code, label in TESS_LANGS.items():
    try:
        t0 = time.time()
        # same downscale as the PaddleOCR path, so timings are comparable
        pages = [pytesseract.image_to_string(downscale(img), lang=code) for img in images]
        text = '\n'.join(p.strip() for p in pages if p.strip())
        tess_text[code] = text
        print(f'\n=== tesseract {label} ({code}) — {time.time() - t0:.1f}s ===')
        print(text[:600] if text else '  (no text found)')
    except Exception as e:
        tess_text[code] = ''
        print(f'\n=== tesseract {label} ({code}) FAILED: {type(e).__name__}: {e} ===')

# Promote Tesseract output only if PaddleOCR gave us nothing for that script.
if not ocr_text.strip() and tess_text.get('eng', '').strip():
    ocr_text = tess_text['eng']
    print('\n[!] PaddleOCR returned no Latin-script text -- using Tesseract eng for ocr_text.')

print(f'\nocr_text now: {ocr_text[:200]}')


=== tesseract English (eng) — 3.2s ===
O10} JO HOA

Hu

     

 

MOU.
Py eee

=== tesseract Hindi (hin) — 3.0s ===
॥ ०030 १४०,

४५०

     

 

४४,
2

=== tesseract Marathi (mar) — 3.8s ===
ग 9001040

09

     

 

॥५%९०१/४,
९४७४६६६.

ocr_text now: INCL. OF ALL TAXES MFG.NOV.2024 EXP.OCT.2026 ER 20 TABS. Paracetamol Tablets IP mol Tablets IP 500mg p nd     e PARACIP-500-500 CIP-500-500 Antipyretic & Relief from Body Pain Taking mpue serious her 


## 3. Expiry / MRP extraction

Same patterns as `app/parsers.py` — keep them in sync if you tune them here.

In [ ]:
import re

date_pat = re.compile(r'(?:EXP|Expiry|Exp\.?)[:\s.]*([A-Z]{3}[.\s/-]?\d{2,4}|\d{1,2}[./-]\d{2,4})', re.I)
mrp_pat = re.compile(r'(?:MRP|Rs\.?|₹)[:\s.]*([\d,.]+)', re.I)

print('OCR text         :', ocr_text[:300])
print('expiry candidates:', date_pat.findall(ocr_text))
print('MRP candidates   :', mrp_pat.findall(ocr_text))

OCR text         : INCL. OF ALL TAXES MFG.NOV.2024 EXP.OCT.2026 ER 20 TABS. Paracetamol Tablets IP mol Tablets IP 500mg p nd     e PARACIP-500-500 CIP-500-500 Antipyretic & Relief from Body Pain Taking mpue serious her com from Body Pain Taling more than daly dose of Paracetamol may cause Each uncoated tablet contains
expiry candidates: ['OCT.2026', 'APR.28']
MRP candidates   : ['10.30']


In [ ]:
## 4. Findings (fill in — goes into the Sem-7 report)

| Check | PaddleOCR | Tesseract |
|---|---|---|
| Ran at all (CPU, no GPU) | | |
| Latency for 3 images | s | s |
| Drug name read correctly? | | |
| EXP date read correctly? | | |
| MRP read correctly? | | |
| Devanagari (Marathi/Hindi) | | |

**Which engine wins?** Record the working PaddleOCR config (the `[ocr_version=..., mkldnn=...]`
tag it prints) so the choice is reproducible. If Tesseract matches or beats PaddleOCR on
these photos, that is a legitimate result — pick the one that actually reads Indian medicine
strips, not the one with the better paper.

**If the drug name was read but the guardrail declined it**, that's expected —
`data/drug_names_seed.txt` is a small placeholder list. Add the drug and re-run to confirm
the match path works, then note that sourcing a real CDSCO-derived drug list is an open
task (see `data/README.md` §4).

**If OCR misread the strip**, capture *why* — glare, foil reflection, curved surface, small
print, low contrast. That failure list drives the M3 data-collection protocol in
`docs/data_collection_guide.md` and justifies any preprocessing step you add later.

---

### Findings banked from this spike

**Process isolation is an architectural constraint.** The VLM stack (PyTorch) and the OCR
stack (PaddlePaddle) cannot share a process — each bundles its own OpenMP runtime, and
co-loading them kills the kernel outright. On the Android port this stops being a notebook
annoyance: `app/modes/` will need either one unified runtime or genuinely separate inference
processes. Worth a line in the report's system-design section.

**Version-matrix fragility is a real project risk.** Getting OCR running required pinning
around three separate upstream breakages: `transformers` v5 breaking `trust_remote_code`
models, `surya-ocr` 2.x moving to a server architecture, and PaddleOCR 3.7.0 + PaddlePaddle
3.3.x crashing on the PIR/oneDNN CPU path (Paddle issue #77340) while PP-OCRv6 silently
dropped Devanagari. Mitigation adopted: explicit pins, runtime introspection instead of
hardcoded API shapes, and a second independent OCR engine as a cross-check. This belongs in
the methodology section — it is the reproducibility argument for the whole project.

**Input resolution is a deployment parameter.** Source photos were 12 MP (4080×3072);
everything downscales to 1600px on the long side before inference. Faster, avoids a reported
PaddleOCR CPU memory blowup, and mirrors what the phone app must do anyway.

SyntaxError: invalid character '—' (U+2014) (3863101929.py, line 14)

## 4. Findings (fill in — goes into the Sem-7 report)

| Check | Result |
|---|---|
| PaddleOCR installs + runs (CPU, no GPU) | |
| Latency per image (CPU) | s |
| Drug name read correctly? | |
| EXP date read correctly? | |
| MRP read correctly? | |
| Devanagari text read correctly? | |
| Guardrail verdict (cell above) | matched / declined |

**If the drug name was read but the guardrail declined it**, that's expected —
`data/drug_names_seed.txt` is a small placeholder list. Add the drug and re-run to confirm
the match path works, then note that sourcing a real CDSCO-derived drug list is an open
task (see `data/README.md` §4).

**If OCR misread the strip**, capture *why* — glare, foil reflection, curved surface, small
print, low contrast. That failure list drives the M3 data-collection protocol in
`docs/data_collection_guide.md` and justifies any preprocessing step you add later.

**Process isolation is itself a finding.** The VLM stack (PyTorch) and the OCR stack
(PaddlePaddle) cannot share a process. On the Android port this stops being a notebook
annoyance and becomes an architectural constraint: the modes in `app/modes/` will need
either a single unified runtime or genuinely separate inference processes. Worth a line in
the report's system-design section.